<a href="https://colab.research.google.com/github/cashmoneysnake/a-b-test/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A/B Test Analysis: Ad vs PSA

## 1. Setup and data

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving marketing_ab_clean.csv to marketing_ab_clean.csv


## 2. Data checks

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

df = pd.read_csv("marketing_ab_clean.csv")
df.head()

,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour
0,1069124,ad,0,130,Monday,20
1,1119715,ad,0,93,Tuesday,22
2,1144181,ad,0,21,Tuesday,18
3,1435133,ad,0,355,Tuesday,10
4,1015700,ad,0,276,Friday,14


## 3. Power analysis

In [ ]:
print("shape:", df.shape)
print("duplicate users:", df["user_id"].duplicated().sum())
print("nulls:", df.isna().sum().sum())
print(df["test_group"].value_counts())

shape: (588101, 6)
duplicate users: 0
nulls: 0
test_group
ad     564577
psa     23524
Name: count, dtype: int64


## 4. Validity: sample ratio and allocation

In [ ]:
ALPHA, POWER, MDE = 0.05, 0.80, 0.003   # fixed

ctrl = df[df["test_group"] == "psa"]
n_ctrl = (df["test_group"] == "psa").sum()
n_trt = (df["test_group"] == "ad").sum()
ratio = n_trt / n_ctrl                   # how many treatment users per control user

baseline = ctrl["converted"].mean()
effect = proportion_effectsize(baseline + MDE, baseline)   # Cohen's h
needed = NormalIndPower().solve_power(effect_size=effect, alpha=ALPHA,
                                      power=POWER, ratio=ratio)

print(f"baseline conversion: {baseline:.4%}")
print(f"control users needed: {needed:,.0f}  |  control users available: {n_ctrl:,}")

baseline conversion: 1.7854%
control users needed: 17,216  |  control users available: 23,524


## 5. Validity: group balance

In [ ]:
counts = df["test_group"].value_counts()
print(counts, (counts / counts.sum()).round(4), sep="\n")

def allocation_check(col):
    table = pd.crosstab(df[col], df["test_group"])            # counts of ad/psa per day or hour
    chi2, p, dof, expected = stats.chi2_contingency(table)    # does group depend on this column?
    n = table.values.sum()
    cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))  # effect size, 0 = none, 1 = perfect link
    psa_share = (table["psa"] / table.sum(axis=1)).round(4)   # control share in each day/hour
    return chi2, p, dof, cramers_v, psa_share

for col in ["most_ads_day", "most_ads_hour"]:
    chi2, p, dof, v, share = allocation_check(col)
    print(f"\n{col}: chi2={chi2:.2f}, dof={dof}, p={p:.4g}, Cramer's V={v:.4f}") #weak association
    print("psa share min/max:", share.min(), share.max())

test_group
ad     564577
psa     23524
Name: count, dtype: int64
test_group
ad     0.96
psa    0.04
Name: count, dtype: float64

most_ads_day: chi2=235.61, dof=6, p=4.849e-48, Cramer's V=0.0200
psa share min/max: 0.035 0.0471

most_ads_hour: chi2=192.29, dof=23, p=1.095e-28, Cramer's V=0.0181
psa share min/max: 0.0301 0.0455


## 6. Primary analysis

In [ ]:
def smd(a, b):   # standardized mean difference: gap in units of spread
    pooled = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return (a.mean() - b.mean()) / pooled

ad  = df.loc[df.test_group == "ad",  "total_ads"]
psa = df.loc[df.test_group == "psa", "total_ads"]
print("mean ad/psa:", round(ad.mean(), 2), round(psa.mean(), 2),
      "| median:", ad.median(), psa.median())
print("SMD:", round(smd(ad, psa), 4))
print("Mann-Whitney p:", stats.mannwhitneyu(ad, psa).pvalue)  # very right skewed

mean ad/psa: 24.82 24.76 | median: 13.0 12.0
SMD: 0.0014
Mann-Whitney p: 4.6909108547205694e-11


## 7. Robustness check

In [ ]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

def primary(d, label):
    t = d[d.test_group == "ad"]["converted"]
    c = d[d.test_group == "psa"]["converted"]
    x = np.array([t.sum(), c.sum()]); n = np.array([len(t), len(c)])
    p_t, p_c = x / n
    z, p = proportions_ztest(x, n)
    lo, hi = confint_proportions_2indep(x[0], n[0], x[1], n[1], method="newcomb", compare="diff")
    print(f"--- {label} ---")
    print(f"treatment (ad): {x[0]:,}/{n[0]:,} = {p_t:.4%}")
    print(f"control (psa):  {x[1]:,}/{n[1]:,} = {p_c:.4%}")
    print(f"z = {z:.2f}, p = {p:.4g}")
    print(f"absolute lift: {(p_t-p_c)*100:.3f} pp, 95% CI [{lo*100:.3f}, {hi*100:.3f}] pp (Newcombe)")
    return len(t), len(c), p_t, p_c

n_t, n_c, p_t, p_c = primary(df, "PRIMARY: all users")

# relative lift CI by bootstrap (10,000 resamples, fixed seed)
rng = np.random.default_rng(42)
bt = rng.binomial(n_t, p_t, 10_000) / n_t
bc = rng.binomial(n_c, p_c, 10_000) / n_c
rel = (bt - bc) / bc
print(f"relative lift: {p_t/p_c-1:.1%}, 95% CI [{np.percentile(rel,2.5):.1%}, {np.percentile(rel,97.5):.1%}]")

--- PRIMARY: all users ---
treatment (ad): 14,423/564,577 = 2.5547%
control (psa):  420/23,524 = 1.7854%
z = 7.37, p = 1.705e-13
absolute lift: 0.769 pp, 95% CI [0.587, 0.936] pp (Newcombe)
relative lift: 43.1%, 95% CI [30.6%, 57.9%]


## 8. Sensitivity: adjusted lift

In [ ]:
p99 = df["total_ads"].quantile(0.99)
primary(df[df.total_ads <= p99], f"Outliercheck: total_ads <= {p99:.0f}");

--- Outliercheck: total_ads <= 202 ---
treatment (ad): 13,554/559,017 = 2.4246%
control (psa):  380/23,274 = 1.6327%
z = 7.75, p = 9.541e-15
absolute lift: 0.792 pp, 95% CI [0.616, 0.952] pp (Newcombe)


## 9. Conclusion:  Ship

In [ ]:
import statsmodels.api as sm
from patsy import dmatrices, dmatrix

df["treat"] = (df.test_group == "ad").astype(int)
# collapse to one row per day/hour/treatment: same model, far less memory
g = (df.groupby(["most_ads_day", "most_ads_hour", "treat"])["converted"]
       .agg(conv="sum", n="size").reset_index())
g["fail"] = g["n"] - g["conv"]

y, X = dmatrices("conv + fail ~ treat + C(most_ads_day) + C(most_ads_hour)", g, return_type="dataframe")
m = sm.GLM(y, X, family=sm.families.Binomial()).fit()
print("odds ratio for treat:", round(np.exp(m.params["treat"]), 3))

# average lift in percentage points: everyone treated vs everyone control, same day/hour mix
cells = g.groupby(["most_ads_day", "most_ads_hour"])["n"].sum().reset_index()
w = (cells["n"] / cells["n"].sum()).values
A1 = np.asarray(dmatrix(X.design_info, cells.assign(treat=1)))
A0 = np.asarray(dmatrix(X.design_info, cells.assign(treat=0)))
def ame(b): return (((1/(1+np.exp(-A1@b))) - (1/(1+np.exp(-A0@b)))) * w).sum()

est = ame(m.params.values)
draws = np.random.default_rng(42).multivariate_normal(m.params.values, m.cov_params().values, 5000)
sims = np.array([ame(b) for b in draws])
print(f"ADJUSTED lift: {est*100:.3f} pp, 95% CI [{np.percentile(sims,2.5)*100:.3f}, {np.percentile(sims,97.5)*100:.3f}] pp")

odds ratio for treat: 1.442
ADJUSTED lift: 0.769 pp, 95% CI [0.590, 0.935] pp
